In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://data.cityofnewyork.us/resource/"
CRASH_ID = "h9gi-nx95"
SUMM_HIST_ID = "sv2w-rv3k"
SUMM_YTD_ID = "mv4k-y93f"
STOPS_ID = "hn9i-dwpr"

START = "2019-01-01"

In [ ]:

def fetch_socrata(dataset_id, select, where, date_col, page=50000):
    params = {"$select": select, "$where": where, "$order": date_col, "$limit": page}
    frames, offset = [], 0
    while True:
        params["$offset"] = str(offset)
        r = requests.get(BASE + dataset_id + ".json", params=params, timeout=90)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        frames.append(pd.DataFrame(batch))
        offset += len(batch)
    df = pd.concat(frames, ignore_index=True)
    df[date_col] = pd.to_datetime(df[date_col])
    return df

print("fetcher ready")

In [ ]:
crashes = fetch_socrata(
    CRASH_ID,
    select="crash_date, borough, number_of_cyclist_injured, number_of_cyclist_killed",
    where="(number_of_cyclist_injured > 0 OR number_of_cyclist_killed > 0) "
          f"AND crash_date >= '{START}'",
    date_col="crash_date",
)
for c in ["number_of_cyclist_injured", "number_of_cyclist_killed"]:
    crashes[c] = crashes[c].astype(int)
crashes["bike_harm"] = crashes["number_of_cyclist_injured"] + crashes["number_of_cyclist_killed"]
print(len(crashes), "cyclist-involved crashes,", crashes["crash_date"].min().date(), "->", crashes["crash_date"].max().date())

In [ ]:
#  Field names below are best-guess placeholders. After the first fetch, run
# `sums_hist.columns` and inspect a few rows, then pin the actual names and the
#  bicycle filter. From your cyclist-summonses work, `LAW_SECTION_NUMBER` was the
#  most reliable bicycle field (fewest nulls) — a groupby on it will surface the
#  bicycle-related sections.

sums_hist = fetch_socrata(
    SUMM_HIST_ID,
    select="SUMMONS_DATE, OFFENSE_DESCRIPTION, LAW_DESCRIPTION, LAW_SECTION_NUMBER",  # EDIT
    where="LOWER(OFFENSE_DESCRIPTION) LIKE '%bike%' "
          "OR LOWER(LAW_DESCRIPTION) LIKE '%bike%'",
    date_col="SUMMONS_DATE",
)
sums_ytd = fetch_socrata(
    SUMM_YTD_ID,
    select="SUMMONS_DATE, OFFENSE_DESCRIPTION, LAW_DESCRIPTION, LAW_SECTION_NUMBER",  # EDIT
    where="LOWER(OFFENSE_DESCRIPTION) LIKE '%bike%' "
          "OR LOWER(LAW_DESCRIPTION) LIKE '%bike%'",
    date_col="SUMMONS_DATE",
)
sums = pd.concat([sums_hist, sums_ytd], ignore_index=True).drop_duplicates()
sums = sums[sums["SUMMONS_DATE"] >= START]
print(len(sums), "bike-related summons,", sums["SUMMONS_DATE"].min().date(), "->", sums["SUMMONS_DATE"].max().date())


# API quirk: some Socrata endpoints reject literal `%` in `$where` passed through
# requests. If a summons query returns zero rows, double-check the column names first
# (a wrong column raises a 400 with a clear message — good for confirming names).

In [ ]:
stops = fetch_socrata(
    STOPS_ID,
    select="STOP_DATETIME, VEHICLE_CATEGORY",   # EDIT
    where="VEHICLE_CATEGORY IN ('BIKE','MOPED','SCOOTER')",   # EDIT values too
    date_col="STOP_DATETIME",
)
stops = stops[stops["STOP_DATETIME"] >= START]
print(len(stops), "bike-ish stops,", stops["STOP_DATETIME"].min().date(), "->", stops["STOP_DATETIME"].max().date())